# Construction Safety Vision — Colab Training
Train a custom PPE detector on a Colab GPU, evaluate it, and export an ONNX artifact for the runtime application. Before running, select **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics==8.3.221 onnx==1.18.0 onnxslim==0.1.71
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

## Dataset
Upload a YOLO-format ZIP to Google Drive. It must contain `images/{train,val,test}`, `labels/{train,val,test}`, and `dataset.yaml`. Keep the test split isolated until final evaluation.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, yaml
drive.mount('/content/drive')
DATASET_ZIP = Path('/content/drive/MyDrive/construction-safety/dataset.zip')
assert DATASET_ZIP.exists(), f'Upload dataset to {DATASET_ZIP}'
shutil.unpack_archive(DATASET_ZIP, '/content/construction-safety-dataset')
DATA_YAML = Path('/content/construction-safety-dataset/dataset.yaml')
print(yaml.safe_load(DATA_YAML.read_text()))

In [ ]:
# Dataset sanity check: missing or orphaned labels should stop the run.
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
for split in ('train', 'val', 'test'):
    images = [p for p in (DATA_YAML.parent/'images'/split).rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS]
    labels = list((DATA_YAML.parent/'labels'/split).glob('*.txt'))
    image_stems, label_stems = {p.stem for p in images}, {p.stem for p in labels}
    print(split, 'images=', len(images), 'labels=', len(labels))
    assert image_stems <= label_stems, f'{split}: images without labels: {sorted(image_stems-label_stems)[:5]}'

In [ ]:
model = YOLO('yolo11s.pt')
results = model.train(
    data=str(DATA_YAML), epochs=80, imgsz=640, batch=-1, device=0,
    patience=15, workers=2, project='/content/runs', name='ppe-yolo11s',
    seed=42, deterministic=True, cache=False, amp=True,
    degrees=5, translate=0.1, scale=0.4, fliplr=0.5, mosaic=0.5, close_mosaic=10,
)

In [ ]:
best = YOLO('/content/runs/ppe-yolo11s/weights/best.pt')
metrics = best.val(data=str(DATA_YAML), split='test', device=0, plots=True)
print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)

In [ ]:
onnx_path = best.export(format='onnx', imgsz=640, opset=17, simplify=True, dynamic=False)
ARTIFACTS = Path('/content/drive/MyDrive/construction-safety/artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/runs/ppe-yolo11s/weights/best.pt', ARTIFACTS/'best.pt')
shutil.copy2(onnx_path, ARTIFACTS/'safety_detector.onnx')
shutil.copytree('/content/runs/ppe-yolo11s', ARTIFACTS/'training-run', dirs_exist_ok=True)
print('Saved artifacts to', ARTIFACTS)